In [ ]:
# Toxic Comment Classification System
## Objective
Build a multiclass NLP model to classify comments into categories like toxic, insult, threat, etc.

In [100]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [106]:
## Data Loading
df = pd.read_csv('/content/archive (11).zip')
# Replace with your dataset path

In [108]:
## Data Cleaning
df.drop(columns=['timestamp','comment_id','user_name'],inplace=True)

In [112]:

df = df.drop_duplicates(subset=['comment_text'])
df = df[['comment_text', 'label']]   # keep only needed columns

In [114]:
## Text Preprocessing
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df['label'])

In [116]:
df['comment_text'] = df['comment_text'].str.lower()

In [118]:

import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [119]:
from nltk.corpus import stopwords


198


In [120]:
import string
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [121]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [122]:
def transfrom_text(text):
  text = nltk.word_tokenize(text)
  y =[]
  for i in text:
    if i.isalnum():
      y.append(i)
  text=y[:]
  y.clear()
  for i in text:
    if i not in stopwords.words('english') and i not in string.punctuation :
      y.append(i)
  text = y[:]
  y.clear()
  for i in text:
    y.append(ps.stem(i))
  return " ".join(y)

In [123]:
!pip install nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [124]:
df['transformed_text'] = df['comment_text'].apply(transfrom_text)

In [126]:
## Feature Extraction (TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df['transformed_text']).toarray()

In [127]:
y = df['label']

In [128]:
## Model Training
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [129]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [130]:
## Evaluation
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

Precision: 0.974677943530053
Recall: 0.972885032537961
F1 Score: 0.9729714687545526


In [132]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("LR Accuracy:", accuracy_score(y_test, y_pred_lr))

LR Accuracy: 0.9812002892263196


In [133]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5)

print("Cross-val scores:", scores)
print("Average:", scores.mean())

Cross-val scores: [0.97686189 0.9703543  0.97541576 0.97975416 0.97505423]
Average: 0.9754880694143168


In [ ]:
## Final Conclusion
The Logistic Regression model performed best with ~98% accuracy.
Cross-validation results confirm that the model generalizes well and does not overfit.